### Set Up

In [4]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pprint
from scipy.stats import chi2_contingency, ttest_ind

### Define Functions 

In [5]:
#  takes in an annotator output and returns a dictionary with the events as keys and the binary labellings as values

def parse_events_with_labels(file_path):
    """
    Parse a JSON file containing event nodes and extract their C/I/K polarity labels.
    
    Args:
        file_path: Path to the JSON file
        
    Returns:
        dict: Mapping of event labels to their C/I/K polarity strings
              e.g., {"Historical buildings are demolished": "C+I+K+", ...}
    """
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    # Parse each line as a separate JSON object
    nodes = [json.loads(line.strip()) for line in lines if line.strip()]
    
    # Find the being node (first node with kind "being")
    being_node = None
    for node_obj in nodes:
        if node_obj.get('node', {}).get('kind') == 'being':
            being_node = node_obj
            break
    
    if not being_node:
        return {}
    
    # Create a mapping from event labels to their C/I/K values
    event_labels = {}
    
    for link in being_node.get('links', []):
        to_node_label = link.get('to_node')
        b_link_value = link.get('link', {}).get('value')
        
        if to_node_label and b_link_value:
            event_labels[to_node_label] = b_link_value
    
    # Filter to only include actual events
    event_nodes = {
        node_obj['node']['label'] 
        for node_obj in nodes 
        if node_obj.get('node', {}).get('kind') == 'event'
    }
    
    return {label: value for label, value in event_labels.items() if label in event_nodes}

### Process Severe Harm Scenarios

In [ ]:
#select the annotated scenario outputs that will be analyzed
annotated_output_path = Path().resolve() / "../../annotated_outputs/franken/conditions_severe_harm_very_good"
print(f"annotated_output_path: {annotated_output_path}")

annotated_output_path: /Users/anna/Dropbox/2025_moral_scenario_annotation/code/anna/tree/graph_extract/analysis/franken/../../annotated_outputs/franken/conditions_severe_harm_very_good


In [7]:
# print out every folder name inside annotated_output_path and print the count of .json files that end with "choice_1.json" in each folder
for folder in annotated_output_path.iterdir():
    if folder.is_dir():
        json_files = list(folder.glob("*choice_1.json"))
        print(f"Folder: {folder.name}, Count of choice_1.json files: {len(json_files)}")

Folder: cc_evitable_action_yes_stories, Count of choice_1.json files: 10
Folder: cc_inevitable_prevention_no_stories, Count of choice_1.json files: 10
Folder: coc_evitable_action_yes_stories, Count of choice_1.json files: 10
Folder: cc_inevitable_action_yes_stories, Count of choice_1.json files: 10
Folder: coc_inevitable_action_yes_stories, Count of choice_1.json files: 10
Folder: coc_evitable_prevention_no_stories, Count of choice_1.json files: 10
Folder: cc_evitable_prevention_no_stories, Count of choice_1.json files: 10


In [8]:
# test the function with some annotator output
test_file = annotated_output_path / "cc_inevitable_prevention_no_stories" / "5_choice_1.json"
pprint.pprint(parse_events_with_labels(test_file))

{"A thorough investigation of the corporation's unethical practices is initiated.": 'C-I-K+',
 'Future employees of the corporation are protected from exploitation.': 'C-I-K-',
 'I avoid direct involvement in the theft of sensitive information.': 'C+I+K+',
 'I prevent the creation of the controversial art piece.': 'C+I+K+',
 'My colleague is unable to express criticism through the controversial art piece.': 'C+I+K+',
 "The public outcry against the corporation is sparked by the whistleblower's actions.": 'C-I-K-',
 'The whistleblower within the corporation still steals highly sensitive information.': 'C-I-K+'}


In [10]:
#  create dataframe with all annotator-generated events for choice 1s of all scenarios in this set

data = []
for folder in annotated_output_path.iterdir():
    if folder.is_dir():
        evitability = "Inevitable" if "inevitable" in folder.name else "Evitable"
        means_side_effect = "CC (Means)" if "cc" in folder.name else "COC (SideEff)"
        co_omission = "Commission" if "action_yes" in folder.name else "Omission"
        
        for json_file in folder.glob("*choice_1.json"):
            sid = json_file.stem.split("_")[0]  # Extract SID from filename
            
            event_labels = parse_events_with_labels(json_file)
            
            for event, cik_value in event_labels.items():
                c_value = cik_value[1]  # C polarity
                i_value = cik_value[3]  # I polarity
                k_value = cik_value[5]  # K polarity
                
                data.append({
                    "SID": sid,
                    "Folder name": folder.name,
                    "Evitability": evitability,
                    "Means/Side Effect": means_side_effect,
                    "Commission/Omission": co_omission,
                    "Event": event,
                    "C": c_value,
                    "I": i_value,
                    "K": k_value
                })

folder_order = [
    "cc_evitable_action_yes_stories",
    "cc_evitable_prevention_no_stories",
    "cc_inevitable_action_yes_stories",
    "cc_inevitable_prevention_no_stories",
    "coc_evitable_action_yes_stories",
    "coc_evitable_prevention_no_stories",
    "coc_inevitable_action_yes_stories",
    "coc_inevitable_prevention_no_stories"
]

# reorder the rows that the SID order is ascending within each folder
data.sort(key=lambda x: (folder_order.index(x["Folder name"]), int(x["SID"])))

In [ ]:
# # print dataframe
# from IPython.display import display, HTML
# df = pd.DataFrame(data)
# display(HTML(df.to_html()))

In [13]:
# define the primary harm of each scenario (completed by hand)
primary_harm_rows_idx = [1, 12, 17, 21, 25, 35, 41, 49, 54, 56,
                        62, 69, 75, 80, 85, 92, 98, 104, 110, 114, 
                        118, 124, 132, 138, 144, 155, 159, 163, 171, 174, 
                        179, 188, 199, 204, 206, 211, 219, 228, 235, 240,
                        249, 255, 258, 260, 272, 280, 285, 291, 295, 298,
                        301, 309, 314, 319, 323, 330, 336, 344, 349, 354,
                        361, 367, 370, 375, 385, 393, 398, 403, 407, 411]

In [14]:
# display the data frame having selected the primary harms only
handpicked_df = df.iloc[primary_harm_rows_idx]
display(HTML(handpicked_df.to_html()))

,SID,Folder name,Evitability,Means/Side Effect,Commission/Omission,Event,C,I,K
1,0,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,Long-established residential community is relocated,+,+,+
12,1,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,Possible health and safety risks during power outage,+,-,+
17,2,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,Students' mental health is affected negatively,+,-,+
21,3,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,The patient experiences a temporary but significant deterioration in health.,+,+,+
25,4,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,Several staff members are dismissed due to lack of training in plant-based cuisine,+,+,+
35,5,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,The corporation experiences reputational damage,+,+,+
41,6,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,Some patients experience irreversible health changes,+,-,+
49,7,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,Laid off employees face job loss and potential financial hardship,+,+,+
54,8,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,The economies of both towns are severely disrupted,+,+,+
56,9,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,The criminal informant continues their illegal activities,+,+,+


### Process Mild Harm Scenarios

In [15]:
#select the annotated scenario outputs that will be analyzed
annotated_output_path = Path().resolve() / "../../annotated_outputs/franken/conditions_mild_harm_mild_good"
print(f"annotated_output_path: {annotated_output_path}")

annotated_output_path: /Users/anna/Dropbox/2025_moral_scenario_annotation/code/anna/tree/graph_extract/analysis/franken/../../annotated_outputs/franken/conditions_mild_harm_mild_good


In [16]:
#  create dataframe with all annotator-generated events for choice 1s of all scenarios in this set

data = []
for folder in annotated_output_path.iterdir():
    if folder.is_dir():
        evitability = "Inevitable" if "inevitable" in folder.name else "Evitable"
        means_side_effect = "CC (Means)" if "cc" in folder.name else "COC (SideEff)"
        co_omission = "Commission" if "action_yes" in folder.name else "Omission"
        
        for json_file in folder.glob("*choice_1.json"):
            sid = json_file.stem.split("_")[0]  # Extract SID from filename
            
            event_labels = parse_events_with_labels(json_file)
            
            for event, cik_value in event_labels.items():
                c_value = cik_value[1]  # C polarity
                i_value = cik_value[3]  # I polarity
                k_value = cik_value[5]  # K polarity
                
                data.append({
                    "SID": sid,
                    "Folder name": folder.name,
                    "Evitability": evitability,
                    "Means/Side Effect": means_side_effect,
                    "Commission/Omission": co_omission,
                    "Event": event,
                    "C": c_value,
                    "I": i_value,
                    "K": k_value
                })

folder_order = [
    "cc_evitable_action_yes_stories",
    "cc_evitable_prevention_no_stories",
    "cc_inevitable_action_yes_stories",
    "cc_inevitable_prevention_no_stories",
    "coc_evitable_action_yes_stories",
    "coc_evitable_prevention_no_stories",
    "coc_inevitable_action_yes_stories",
    "coc_inevitable_prevention_no_stories"
]

# reorder the rows that the SID order is ascending within each folder
data.sort(key=lambda x: (folder_order.index(x["Folder name"]), int(x["SID"])))

In [17]:
# ADD HERE: SELECTION OF PRIMARY HARMS